MODULE
1. Encoder
2. Token_Representation -> Ent,Span
3. Entity FFN
4. Span FFN
5. Matching
6. Loss
7. Decder

In [3]:
BATCH_SIZE=16
SPAN_CHANNEL=100

In [4]:
# Dataset
import json
import torch
from torch.utils.data import Dataset, DataLoader

class GLiNERDataset(Dataset):
    def __init__(self, path):
        self.data = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                self.data.append(json.loads(line))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        return {
        "input_ids": torch.tensor(item["input_ids"], dtype=torch.long),
        "attention_mask": torch.tensor(item["attention_mask"], dtype=torch.long),
        "text_mask": torch.tensor(item["text_mask"], dtype=torch.long),
        "ent_mask": torch.tensor(item["ent_mask"], dtype=torch.long),
        "word_index": torch.tensor(item["word_index"], dtype=torch.long),
        "span_labels": item["span_labels"]
}

In [5]:
def gliner_collate_fn(batch):
    max_len = max(len(x["input_ids"]) for x in batch)

    input_ids_list = []
    attention_mask_list = []
    ent_mask_list = []
    text_mask_list = []
    span_labels_list = []

    for item in batch:
        seq_len = len(item["input_ids"])
        pad_len = max_len - seq_len

        input_ids = torch.cat([
            item["input_ids"],
            torch.zeros(pad_len, dtype=torch.long)
        ], dim=0)

        attention_mask = torch.cat([
            item["attention_mask"],
            torch.zeros(pad_len, dtype=torch.long)
        ], dim=0)

        ent_mask = torch.cat([
            item["ent_mask"],
            torch.zeros(pad_len, dtype=torch.bool)
        ], dim=0)

        text_mask = torch.cat([
            item["text_mask"],
            torch.zeros(pad_len, dtype=torch.bool)
        ], dim=0)

        input_ids_list.append(input_ids)
        attention_mask_list.append(attention_mask)
        ent_mask_list.append(ent_mask)
        text_mask_list.append(text_mask)
        span_labels_list.append(item["span_labels"])

    word_index = torch.nn.utils.rnn.pad_sequence(
        [b["word_index"] for b in batch],
        batch_first=True,
        padding_value=-1
    )

    return {
        "input_ids": torch.stack(input_ids_list, dim=0),
        "attention_mask": torch.stack(attention_mask_list, dim=0),
        "text_mask": torch.stack(text_mask_list, dim=0),
        "ent_mask": torch.stack(ent_mask_list, dim=0),
        "word_index": word_index,
        "span_labels": span_labels_list
    }

In [6]:
# Dataloader
train_dataset = GLiNERDataset("train_gliner-3.jsonl")
val_dataset = GLiNERDataset("val_gliner.jsonl")

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=gliner_collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=gliner_collate_fn
)

In [7]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer,AutoModel

In [8]:
#Encoder
BackBone_Model="klue/bert-base"

In [9]:
# Input -> (BATCH_SIZE,SEQ_LEN) , Each -> (SEQ_LEN,)
# Output -> (BATCH_SIZE,SEQ_LEN,Hidden_state)
class EncoderModule(nn.Module):
    def __init__(self,backbone_model:str):
        super().__init__()
        self.encoder=AutoModel.from_pretrained(backbone_model)
        self.hidden_size=self.encoder.config.hidden_size

    def forward(self,input_ids:torch.Tensor,attention_mask:torch.Tensor)->torch.Tensor:
        outputs=self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        hidden_states=outputs.last_hidden_state

        return hidden_states

In [10]:
# Token_Representation(h generator)
# Input -> (BATCH_SIZE,SEQ_LEN,Hidden_size) , (BATCH_SIZE,SEQ_LEN)
# Output -> (BATCH_SIZE,T_MAX,Hidden_size)

# 이거는 사용하면 안됨
# 우선 이거는 subtoken이 반영안됨 encoder의 출력은 subtoken으로나옴
# 이 코드는 subtoken단위로 임베딩임 -> 정답지와 좌표가 달라짐
# 때문에 pooling으로 서브토큰을 합쳐야함 그냥 mask 처리하면 안됨
class Text_Token_Represntation(nn.Module):
    def __init__(self):
        super().__init__()

    def pad_sequence(self,sentences):
        max_len=max(line.size(0) for line in sentences)
        hidden_size=sentences[0].size(1)

        pad_sequences=[]

        for sent in sentences:
            pad_len=max_len-sent.size(0)

            if pad_len>0:
                pad_tensor=torch.zeros(
                    pad_len,
                    hidden_size,
                    device=sent.device,
                    dtype=sent.dtype
                )
                sent=torch.cat([sent,pad_tensor],dim=0)

            pad_sequences.append(sent)
        return torch.stack(pad_sequences,dim=0)

    def forward(self,hidden_states,text_mask):
        text_vector=[]

        for hs,mask in zip(hidden_states,text_mask):
            h_vector=hs[mask.bool()]
            text_vector.append(h_vector)

        padded_text_vectors=self.pad_sequence(text_vector)

        return padded_text_vectors

In [11]:
# Pooling 할려면 subtoken인지 알아야하는데 이는 encoder에서 나온word_index를 보고 판단하여 pooling함
# mena,first -> pooling 방식 선택
class WordToken_Representation(nn.Module):
    def __init__(self,pooling="mean"):
        super().__init__()
        assert pooling in ["mean","first"]
        self.pooling=pooling

    def forward(self,hidden_states,text_mask,word_index):
        batch_word_embeddings=[]
        word_len=[]

        B,L,H=hidden_states.shape

        for b in range(B):
            hs=hidden_states[b]
            tm=text_mask[b]
            wi=word_index[b]

            valid_pos=(tm==1) & (wi>=0)

            hs_valid=hs[valid_pos]
            wi_valid=wi[valid_pos]

            max_word_idx=int(wi_valid.max().item())
            num_words=max_word_idx+1

            word_vecs=[]
            for word_id in range(num_words):
                token_vecs=hs_valid[wi_valid==word_id]

                if self.pooling=="mean":
                    word_vec=token_vecs.mean(dim=0)
                elif self.pooling=="first":
                    word_vec=token_vecs[0]
                word_vecs.append(word_vec)

            word_vecs=torch.stack(word_vecs,dim=0)
            batch_word_embeddings.append(word_vecs)
            word_len.append(word_vecs.size(0))

        max_len=max(word_len)
        padded=[]

        for x in batch_word_embeddings:
            pad_len=max_len-x.size(0)
            if pad_len>0:
                pad=torch.zeros(pad_len,x.size(1),device=x.device,dtype=x.dtype)
                x=torch.cat([x,pad],dim=0)
            padded.append(x)

        padded_word_embeddings=torch.stack(padded,dim=0)
        word_len=torch.tensor(word_len,device=hidden_states.device,dtype=torch.long)

        return padded_word_embeddings,word_len

In [12]:
# Token_Representation(p generator)
# Input -> (BATCH_SIZE,SEQ_LEN,Hidden_size) , (BATCH_SIZE,SEQ_LEN)
# Output -> (BATCH_SIZE,M,Hidden_size) (M=[ENT] count)
# batch 단위로 돌리려면 p_vector 길이 같아야함
class ENT_Token_Representation(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self,hidden_states,ent_mask):
        ent_vectors=[]

        for hs,mask in zip(hidden_states,ent_mask):
            p_vector=hs[mask.bool()]
            ent_vectors.append(p_vector)
        max_ent=max(v.size(0) for v in ent_vectors)
        hidden_size=hidden_states.size(-1)

        padded_ent_vectors=[]
        padded_ent_masks=[]

        for v in ent_vectors:
            m=v.size(0)
            pad_len=max_ent-m

            if pad_len>0:
                pad_tensor=torch.zeros(
                    pad_len,
                    hidden_size,
                    device=v.device,
                    dtype=v.dtype
                )
                v=torch.cat([v,pad_tensor],dim=0)
            valid_mask=torch.cat([
                torch.ones(m,device=v.device,dtype=torch.bool),
                torch.zeros(pad_len,device=v.device,dtype=torch.bool)
            ],dim=0)
            padded_ent_vectors.append(v)
            padded_ent_masks.append(valid_mask)

        ent_vectors=torch.stack(padded_ent_vectors,dim=0)
        ent_valid_mask=torch.stack(padded_ent_masks,dim=0)

        return ent_vectors,ent_valid_mask

In [13]:
#class Linear(nn.Module):
#    def __init__(self,input_x,output_y,bias=True):
#        super().__init__()
#
#        self.w=nn.Parameter(torch.randn(output_y,input_x))

 #       if bias:
  #          self.bias=nn.Parameter(torch.randn(output_y))
   #     else:
    #        self.bias=None
    #def forward(self,x):
     #   out=x @ self.w.T

      #  if self.bias is not None:
       #     out+=self.bias
        #return out
# 직접 만든 Linear를 사용하면 torch.randn으로 인하여 초기화가 거칠어져 최적화가 안됨

In [14]:
class Linear(nn.Module):
    def __init__(self, input_x, output_y, bias=True):
        super().__init__()
        self.linear = nn.Linear(input_x, output_y, bias=bias)

    def forward(self, x):
        return self.linear(x)

In [15]:
#class ReLU(nn.Module):
#    def __init__(self):
#        super().__init__()
#    def forward(self,x):
#        if(x>=0):
#            return x
#        else:
#            return 0
# 이거는 Tensor 각각을 보지않아서 동작안함
class ReLU(nn.Module):
    def forward(self,x):
        return x*(x>0)

In [16]:
class Sequential(nn.Module):
    def __init__(self,*layers):
        super().__init__()
        self.layers=nn.ModuleList(layers)

    def forward(self,x):
        for layer in self.layers:
            x=layer(x)

        return x

In [17]:
# Entity FFN
# input -> (Batch_size,M,d_model)
# output -> (Batch_size,M,d_model)
class Entity_Representation(nn.Module):
    def __init__(self,d_model,d_ff):
        super().__init__()

        self.net=Sequential(
            Linear(d_model,d_ff),
            ReLU(),
            Linear(d_ff,d_model)
        )
    def forward(self,x):
        output=self.net(x)
        return output

In [18]:
# Span FFN
# input-> start : (B,H) , end : (B,H)
#
class Span_Representation(nn.Module):
    def __init__(self,d_model,d_ff):
        super().__init__()
        self.net=Sequential(
            Linear(2*d_model,d_ff),
            ReLU(),
            Linear(d_ff,d_model)
        )

    def forward(self,start_token,end_token):
        span=torch.cat([start_token,end_token],dim=-1)
        output=self.net(span)
        return output

In [19]:
class Sigmoid(nn.Module):
    def forward(self,x):
        output=1/(1+torch.exp(-x))
        return output

In [20]:
# sigmoid를 만들어 BCELoss를 쓰면 연산 불안정해짐, 그래서 BCEWithLogitLoss 써야함
# input -> entity : (B,E,H) span : (B,N_span,H) E=M
# output -> logits(B,N_span,E)
class Matching(nn.Module):
    def __init__(self):
        super().__init__()
        self.sigmoid=Sigmoid()
    def forward(self,entity,span):
        entity=entity.transpose(-1,-2)
        output=span@entity
        #score=self.sigmoid(output)
        return output

In [21]:
# flat -> 겹치는 단어 허용 x
# nested -> 중첩된 entity 허용은 하나 부분 겹침만 금지
# 겹치면 True
class Decoder(nn.Module):
    def __init__(self,entity_types,threshold=0.5,mode="flat"):
        super().__init__()
        self.entity_types=entity_types
        self.threshold=threshold
        self.mode=mode

    def overlapping(self,span1,span2):
        s1,e1=span1
        s2,e2=span2

        return not(e1<s2 or e2<s1)

    def partial_overlap(self,span1,span2):
        s1,e1=span1
        s2,e2=span2

        overlap=not(e1<s2 or e2<s1)
        if not overlap:
            return False

        nested=(s1<=s2 and e2<=e1) or (s2<=s1 and e1 <=e2)

        return not nested

    def decode_one(self,probs,span_indices):
        candiates=[]

        n_span,n_ent=probs.shape

        for n in range(n_span):
            s,e=span_indices[n]

            for ent_idx in range(n_ent):
                score=probs[n, ent_idx].item()

                if score>self.threshold:
                    candiates.append({
                        "start":s,
                        "end":e,
                        "label":self.entity_types[ent_idx],
                        "score":score
                    })

        candiates.sort(key=lambda x:x["score"],reverse=True)

        selected=[]

        for cand in candiates:
            c_span=(cand["start"],cand["end"])
            conflict=False

            for sel in selected:
                p_span=(sel["start"],sel["end"])

                if self.mode=="flat":
                    if self.overlapping(c_span,p_span):
                        conflict=True
                        break

                elif self.mode=="nested":
                    if self.partial_overlap(c_span,p_span):
                        conflict=True
                        break

            if not conflict:
                selected.append(cand)

        return selected

    def decode_batch(self,probs_batch,span_indices_batch):
        results=[]

        for probs,span_indices in zip(probs_batch,span_indices_batch):
            results.append(self.decode_one(probs,span_indices))

        return results

In [22]:
class GLiNER(nn.Module):
    def __init__(self,backbone_model,entity_types,d_ff,max_span_width=10,threshold=0.5,mode="flat"):
        super().__init__()

        self.entity_types=entity_types
        self.num_entity_types=len(entity_types)
        self.max_span_width=max_span_width

        self.encoder=EncoderModule(backbone_model)
        d_model=self.encoder.hidden_size

        self.word_rep=WordToken_Representation(pooling="mean")
        self.ent_token_rep=ENT_Token_Representation()

        self.ent_rep=Entity_Representation(d_model,d_ff)
        self.span_rep=Span_Representation(d_model,d_ff)

        self.sigmoid=Sigmoid()
        self.matching=Matching()

        self.decoder=Decoder(
            entity_types=entity_types,
            threshold=threshold,
            mode=mode
        )

    def build_span_representation(self,word_embeddings,word_len):
        B,_,H=word_embeddings.shape

        batch_span_vectors=[]
        batch_span_indices=[]

        for b in range(B):
            w_len=word_len[b].item()
            cur_words=word_embeddings[b]

            starts=[]
            ends=[]
            span_indices=[]

            for start in range(w_len):
                max_end=min(w_len,start+self.max_span_width)
                for end in range(start,max_end):
                    starts.append(start)
                    ends.append(end)
                    span_indices.append((start,end))

            if len(span_indices)==0:
                span_vecs=torch.zeros(0,H,device=word_embeddings.device,dtype=word_embeddings.dtype)
            else:
                start_embs=cur_words[starts]
                end_embs=cur_words[ends]
                span_vecs=self.span_rep(start_embs,end_embs)

            batch_span_vectors.append(span_vecs)
            batch_span_indices.append(span_indices)

        return batch_span_vectors,batch_span_indices

    def forward(self,input_ids,attention_mask,text_mask,ent_mask,word_index):
        hidden_states=self.encoder(input_ids,attention_mask)

        word_embeddings,word_len=self.word_rep(hidden_states=hidden_states,text_mask=text_mask,word_index=word_index)

        ent_embeddings,ent_valid_mask=self.ent_token_rep(hidden_states,ent_mask)
        ent_vectors=self.ent_rep(ent_embeddings)

        span_vectors_batch,span_indices_batch=self.build_span_representation(word_embeddings,word_len)

        logits_batch=[]
        for b in range(input_ids.size(0)):
            span_vec=span_vectors_batch[b]
            ent_vec=ent_vectors[b].unsqueeze(0)
            logits=self.matching(ent_vec,span_vec).squeeze(0)
            logits_batch.append(logits)

        return logits_batch,span_indices_batch

    @torch.no_grad()
    def predict(self,input_ids,attention_mask,text_mask,ent_mask,word_index):
        self.eval()
        logits_batch,span_indices_batch=self.forward(input_ids=input_ids,attention_mask=attention_mask,text_mask=text_mask,ent_mask=ent_mask,word_index=word_index)

        probs_batch=[self.sigmoid(x) for x in logits_batch]
        results=self.decoder.decode_batch(probs_batch,span_indices_batch)

        return results

In [23]:
def build_targets_for_batch(logits_batch, span_indices_batch, span_labels_batch, num_entity_types, device):
    target_batch = []

    for logits, span_indices, gold_labels in zip(logits_batch, span_indices_batch, span_labels_batch):
        n_span = logits.size(0)
        target = torch.zeros((n_span, num_entity_types), device=device)

        span_to_idx = {span: i for i, span in enumerate(span_indices)}

        for s, e, ent_idx in gold_labels:
            if (s, e) in span_to_idx and 0 <= ent_idx < num_entity_types:
                target[span_to_idx[(s, e)], ent_idx] = 1.0

        target_batch.append(target)

    return target_batch

In [24]:
max_label_id = -1

for sample in train_dataset:
    for start, end, label_id in sample["span_labels"]:
        if label_id > max_label_id:
            max_label_id = label_id

print("max_label_id:", max_label_id)
print("num_entity_types:", max_label_id + 1)

max_label_id: 5
num_entity_types: 6


In [25]:
num_entity_types = max_label_id + 1
entity_types_fake = [f"LABEL_{i}" for i in range(num_entity_types)]
entity_types=["PS", "LC", "OG", "DT", "TI", "QT"]
print(entity_types)

['PS', 'LC', 'OG', 'DT', 'TI', 'QT']


In [26]:
model = GLiNER(
    backbone_model=BackBone_Model,
    entity_types=entity_types,
    d_ff=256,
    max_span_width=12,
    threshold=0.5,
    mode="flat"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: klue/bert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [27]:
# 성능개선
# 1. learning rate(가중치를 한번에 얼마나 크게 수정?)
# 2. max_span_width setting
# 3. d_ff dimension
# 4. activation

In [28]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Total params:", total_params)
print("Trainable params:", trainable_params)

Total params: 111602432
Trainable params: 111602432


In [29]:
print(model)

GLiNER(
  (encoder): EncoderModule(
    (encoder): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(32000, 768, padding_idx=0)
        (position_embeddings): Embedding(512, 768)
        (token_type_embeddings): Embedding(2, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): BertEncoder(
        (layer): ModuleList(
          (0-11): 12 x BertLayer(
            (attention): BertAttention(
              (self): BertSelfAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features=768, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): BertSelfOutput(
                (dense): Linear(in_features=768, out_features=768, bias=True)
       

In [30]:
total_params = sum(p.numel() for p in model.parameters())
param_size_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
buffer_size_bytes = sum(b.numel() * b.element_size() for b in model.buffers())

total_size_mb = (param_size_bytes + buffer_size_bytes) / 1024**2

print("Total params:", total_params)
print(f"Param size: {param_size_bytes / 1024**2:.2f} MB")
print(f"Buffer size: {buffer_size_bytes / 1024**2:.2f} MB")
print(f"Total model size: {total_size_mb:.2f} MB")

Total params: 111602432
Param size: 425.73 MB
Buffer size: 0.01 MB
Total model size: 425.74 MB


In [31]:
def compute_span_f1(all_preds, all_golds, entity_types):
    tp = 0
    fp = 0
    fn = 0

    for pred_sample, gold_sample in zip(all_preds, all_golds):
        pred_set = set()
        for item in pred_sample:
            pred_set.add((item["start"], item["end"], item["label"]))

        gold_set = set()
        for start, end, label_id in gold_sample:
            label_name = entity_types[label_id]
            gold_set.add((start, end, label_name))

        tp += len(pred_set & gold_set)
        fp += len(pred_set - gold_set)
        fn += len(gold_set - pred_set)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    return precision, recall, f1, tp, fp, fn

In [32]:
import torch
import torch.nn as nn
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

loss_fn = nn.BCEWithLogitsLoss(reduction="sum")
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

num_epochs = 5
best_f1 = 0.0

for epoch in range(num_epochs):
    print(f"\n===== Epoch {epoch+1}/{num_epochs} =====")

    # ------------------- train -------------------
    model.train()
    total_train_loss = 0.0

    train_bar = tqdm(train_loader, desc="[Train]")

    for batch in train_bar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        text_mask = batch["text_mask"].to(device)
        ent_mask = batch["ent_mask"].to(device)
        word_index = batch["word_index"].to(device)

        optimizer.zero_grad()

        logits_batch, span_indices_batch = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            text_mask=text_mask,
            ent_mask=ent_mask,
            word_index=word_index
        )

        target_batch = build_targets_for_batch(
            logits_batch=logits_batch,
            span_indices_batch=span_indices_batch,
            span_labels_batch=batch["span_labels"],
            num_entity_types=model.num_entity_types,
            device=device
        )

        total_loss = torch.tensor(0.0, device=device)
        total_count = 0

        for logits, target in zip(logits_batch, target_batch):
            if logits.numel() == 0:
                continue
            total_loss = total_loss + loss_fn(logits, target.float())
            total_count += target.numel()

        if total_count == 0:
            continue

        loss = total_loss / total_count
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()
        train_bar.set_postfix({"loss": f"{loss.item():.4f}"})

    avg_train_loss = total_train_loss / len(train_loader)

    # ------------------- valid -------------------
    model.eval()
    total_val_loss = 0.0
    all_preds = []
    all_golds = []

    valid_bar = tqdm(val_loader, desc="[Valid]")

    with torch.no_grad():
        for batch in valid_bar:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            text_mask = batch["text_mask"].to(device)
            ent_mask = batch["ent_mask"].to(device)
            word_index = batch["word_index"].to(device)

            logits_batch, span_indices_batch = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                text_mask=text_mask,
                ent_mask=ent_mask,
                word_index=word_index
            )

            target_batch = build_targets_for_batch(
                logits_batch=logits_batch,
                span_indices_batch=span_indices_batch,
                span_labels_batch=batch["span_labels"],
                num_entity_types=model.num_entity_types,
                device=device
            )

            total_loss = torch.tensor(0.0, device=device)
            total_count = 0

            for logits, target in zip(logits_batch, target_batch):
                if logits.numel() == 0:
                    continue
                total_loss = total_loss + loss_fn(logits, target.float())
                total_count += target.numel()

            if total_count == 0:
                continue

            loss = total_loss / total_count
            total_val_loss += loss.item()

            probs_batch = [torch.sigmoid(x) for x in logits_batch]
            pred_batch = model.decoder.decode_batch(probs_batch, span_indices_batch)

            all_preds.extend(pred_batch)
            all_golds.extend(batch["span_labels"])

            valid_bar.set_postfix({"loss": f"{loss.item():.4f}"})

    avg_val_loss = total_val_loss / len(val_loader)

    precision, recall, f1, tp, fp, fn = compute_span_f1(
        all_preds=all_preds,
        all_golds=all_golds,
        entity_types=model.entity_types
    )

    print(f"Train Loss : {avg_train_loss:.4f}")
    print(f"Valid Loss : {avg_val_loss:.4f}")
    print(f"Precision  : {precision:.4f}")
    print(f"Recall     : {recall:.4f}")
    print(f"F1 Score   : {f1:.4f}")
    print(f"TP / FP / FN = {tp} / {fp} / {fn}")

    if f1 > best_f1:
        best_f1 = f1
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "entity_types": model.entity_types,
                "threshold": model.decoder.threshold,
                "max_span_width": model.max_span_width,
            },
            "best_gliner.pt"
        )
        print("✅ best_gliner.pt 저장 완료")

print("\n학습 종료")
print(f"Best Valid F1: {best_f1:.4f}")


===== Epoch 1/5 =====


[Valid]: 100%|██████████| 313/313 [06:17<00:00,  1.21s/it, loss=0.0014]


Train Loss : 0.0025
Valid Loss : 0.0014
Precision  : 0.7888
Recall     : 0.6461
F1 Score   : 0.7104
TP / FP / FN = 9212 / 2467 / 5045
✅ best_gliner.pt 저장 완료

===== Epoch 2/5 =====


[Valid]: 100%|██████████| 313/313 [06:17<00:00,  1.20s/it, loss=0.0012]


Train Loss : 0.0010
Valid Loss : 0.0011
Precision  : 0.8257
Recall     : 0.7342
F1 Score   : 0.7772
TP / FP / FN = 10467 / 2210 / 3790
✅ best_gliner.pt 저장 완료

===== Epoch 3/5 =====


...

Train Loss : 0.0007
Valid Loss : 0.0009
Precision  : 0.8456
Recall     : 0.7889
F1 Score   : 0.8162
TP / FP / FN = 11247 / 2054 / 3010
✅ best_gliner.pt 저장 완료

===== Epoch 4/5 =====


[Valid]: 100%|██████████| 313/313 [06:17<00:00,  1.21s/it, loss=0.0008]


Train Loss : 0.0006
Valid Loss : 0.0008
Precision  : 0.8622
Recall     : 0.8105
F1 Score   : 0.8356
TP / FP / FN = 11555 / 1846 / 2702
✅ best_gliner.pt 저장 완료

===== Epoch 5/5 =====


[Valid]: 100%|██████████| 313/313 [06:16<00:00,  1.20s/it, loss=0.0008]


Train Loss : 0.0005
Valid Loss : 0.0008
Precision  : 0.8668
Recall     : 0.8167
F1 Score   : 0.8410
TP / FP / FN = 11643 / 1789 / 2614
✅ best_gliner.pt 저장 완료

학습 종료
Best Valid F1: 0.8410


In [33]:
class PT_BioOutpuut:
    def __init__(self,checkpoint_path,backbone_model,d_ff=256,mode="flat",device=None):
        self.device=device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

        checkpoint=torch.load(checkpoint_path,map_location=device)

        entity_types=checkpoint["entity_types"]
        threshold=checkpoint["threshold"]
        max_span_width=checkpoint["max_span_width"]

        self.sigmoid=Sigmoid()

        self.model=GLiNER(
            backbone_model=backbone_model,
            entity_types=entity_types,
            d_ff=d_ff,
            max_span_width=max_span_width,
            threshold=threshold,
            mode=mode
        ).to(self.device)

        self.model.load_state_dict(checkpoint["model_state_dict"])
        self.model.eval()

    def spans_to_bio(self,pred_spans,seq_len):
        bio_tags=["O"]*seq_len

        for item in pred_spans:
            start=item["start"]
            end=item["end"]
            label=item["label"]

            if start<0 or end>=seq_len or start>end:
                continue

            conflict=False
            for i in range(start,end+1):
                if bio_tags[i]!="O":
                    conflict=True
                    break

            if conflict:
                continue
            bio_tags[start]=f"B-{label}"
            for i in range(start+1,end+1):
                bio_tags[i]=f"I-{label}"
        return bio_tags

    def get_word_seq_len(self,word_index_1d):
        valid=word_index_1d[word_index_1d>=0]
        if valid.numel()==0:
            return 0
        return int(valid.max().item())+1

    @torch.no_grad()
    def predict_bio_one(self,input_ids,attention_mask,text_mask,ent_mask,word_index):
        input_ids=input_ids.to(self.device)
        attention_mask=attention_mask.to(self.device)
        text_mask=text_mask.to(self.device)
        ent_mask=ent_mask.to(self.device)
        word_index=word_index.to(self.device)

        logits_batch,span_indices_batch=self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            text_mask=text_mask,
            ent_mask=ent_mask,
            word_index=word_index
        )

        probs_batch=[self.sigmoid(x) for x in logits_batch]
        probs_batch=self.model.decoder.decode_batch(probs_batch,span_indices_batch)

        pred_spans=probs_batch[0]
        seq_len=self.get_word_seq_len(word_index[0])
        bio_tags=self.spans_to_bio(pred_spans,seq_len)

        return bio_tags

    @torch.no_grad()
    def predict_span_and_bio_one(self,input_ids,attention_mask,text_mask,ent_mask,word_index):
        input_ids=input_ids.to(self.device)
        attention_mask=attention_mask.to(self.device)
        text_mask=text_mask.to(self.device)
        ent_mask=ent_mask.to(self.device)
        word_index=word_index.to(self.device)

        logits_batch,span_indices_batch=self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            text_mask=text_mask,
            ent_mask=ent_mask,
            word_index=word_index
            )

        probs_batch=[self.sigmoid(x) for x in logits_batch]
        pred_batch=self.model.decoder.decode_batch(probs_batch,span_indices_batch)

        pred_spans=pred_batch[0]
        seq_len=self.get_word_seq_len(word_index[0])
        bio_tags=self.spans_to_bio(pred_spans,seq_len)

        return pred_spans,bio_tags

In [34]:
Bio_predictor=PT_BioOutpuut(
    checkpoint_path="best_gliner.pt",
    backbone_model=BackBone_Model,
    d_ff=256,
    mode="flat"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: klue/bert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [35]:
pred_spans, bio_tags = Bio_predictor.predict_span_and_bio_one(
    input_ids=batch["input_ids"][0].unsqueeze(0),
    attention_mask=batch["attention_mask"][0].unsqueeze(0),
    text_mask=batch["text_mask"][0].unsqueeze(0),
    ent_mask=batch["ent_mask"][0].unsqueeze(0),
    word_index=batch["word_index"][0].unsqueeze(0)
)

print(pred_spans)
print(bio_tags)

[{'start': 25, 'end': 26, 'label': 'QT', 'score': 0.99953293800354}]
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-QT', 'I-QT', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


In [36]:
model.entity_types=entity_types
model.decoder.entity_types=entity_types

In [42]:
eval_dataset = GLiNERDataset("eval_gliner_1000.jsonl")
eval_loader = DataLoader(
    eval_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=gliner_collate_fn
)

In [46]:
def compute_span_f1(all_preds, all_golds, entity_types):
    tp = 0
    fp = 0
    fn = 0

    for pred_sample, gold_sample in zip(all_preds, all_golds):
        pred_set = set()
        for item in pred_sample:
            pred_set.add((item["start"], item["end"], item["label"]))

        gold_set = set()
        for start, end, label_id in gold_sample:
            label_name = entity_types[label_id]
            gold_set.add((start, end, label_name))

        tp += len(pred_set & gold_set)
        fp += len(pred_set - gold_set)
        fn += len(gold_set - pred_set)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    return precision, recall, f1, tp, fp, fn

model.eval()

all_preds = []
all_golds = []

with torch.no_grad():
    for batch_idx, batch in enumerate(eval_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        text_mask = batch["text_mask"].to(device)
        ent_mask = batch["ent_mask"].to(device)
        word_index = batch["word_index"].to(device)

        logits_batch, span_indices_batch = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            text_mask=text_mask,
            ent_mask=ent_mask,
            word_index=word_index
        )

        probs_batch = [torch.sigmoid(logits) for logits in logits_batch]
        pred_batch = model.decoder.decode_batch(probs_batch, span_indices_batch)

        all_preds.extend(pred_batch)
        all_golds.extend(batch["span_labels"])

        if batch_idx == 0:
            print("첫 배치 예측 예시:", pred_batch[0][:10])
            print("첫 배치 정답 예시:", batch["span_labels"][0][:10])

precision, recall, f1, tp, fp, fn = compute_span_f1(all_preds, all_golds, entity_types)

print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")
print(f"TP={tp}, FP={fp}, FN={fn}")

첫 배치 예측 예시: [{'start': 26, 'end': 27, 'label': 'QT', 'score': 0.941462516784668}, {'start': 4, 'end': 5, 'label': 'DT', 'score': 0.8804888129234314}]
첫 배치 정답 예시: [[4, 5, 3]]
Precision: 0.6728
Recall   : 0.3969
F1       : 0.4993
TP=1178, FP=573, FN=1790


In [47]:
import torch

ckpt = torch.load("best_gliner.pt", map_location="cpu")
print(ckpt.keys())

dict_keys(['model_state_dict', 'entity_types', 'threshold', 'max_span_width'])


In [48]:
for name, param in ckpt["model_state_dict"].items():
    print(name)
    print(param[:3])   # 앞부분만
    break

encoder.encoder.embeddings.word_embeddings.weight
tensor([[-0.0117, -0.0293,  0.1732,  ..., -0.0210, -0.0632, -0.0329],
        [-0.0585,  0.0035, -0.0449,  ..., -0.0077, -0.0162,  0.0634],
        [-0.0055, -0.0278,  0.1989,  ..., -0.0228, -0.0468, -0.0350]])


In [69]:
sample_text = "김도영은 대구에서 4홈런을 쳤다."
sample_tokens = list(sample_text)
print(sample_tokens)

['김', '도', '영', '은', ' ', '대', '구', '에', '서', ' ', '4', '홈', '런', '을', ' ', '쳤', '다', '.']


In [66]:
from transformers import AutoTokenizer
import torch

entity_types = ["PS", "LC", "OG", "DT", "TI", "QT"]

tokenizer = AutoTokenizer.from_pretrained(BackBone_Model)

def build_inference_features(tokens, tokenizer, entity_types, max_length=512):
    text_enc = tokenizer(
        tokens,
        is_split_into_words=True,
        add_special_tokens=False
    )

    text_input_ids = text_enc["input_ids"]
    text_word_ids = text_enc.word_ids()

    entity_input_ids = []
    entity_ent_mask = []

    for ent in entity_types:
        ent_enc = tokenizer(ent, add_special_tokens=False)
        ent_ids = ent_enc["input_ids"]

        for j, tid in enumerate(ent_ids):
            entity_input_ids.append(tid)
            entity_ent_mask.append(1 if j == 0 else 0)

    cls_id = tokenizer.cls_token_id
    sep_id = tokenizer.sep_token_id
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0

    input_ids = [cls_id] + text_input_ids + [sep_id] + entity_input_ids + [sep_id]
    attention_mask = [1] * len(input_ids)
    text_mask = [0] + [1] * len(text_input_ids) + [0] + [0] * len(entity_input_ids) + [0]
    ent_mask = [0] + [0] * len(text_input_ids) + [0] + entity_ent_mask + [0]
    word_index = [-1] + [wid if wid is not None else -1 for wid in text_word_ids] + [-1] + [-1] * len(entity_input_ids) + [-1]

    if len(input_ids) > max_length:
        input_ids = input_ids[:max_length]
        attention_mask = attention_mask[:max_length]
        text_mask = text_mask[:max_length]
        ent_mask = ent_mask[:max_length]
        word_index = word_index[:max_length]

    return {
        "input_ids": torch.tensor([input_ids], dtype=torch.long),
        "attention_mask": torch.tensor([attention_mask], dtype=torch.long),
        "text_mask": torch.tensor([text_mask], dtype=torch.long),
        "ent_mask": torch.tensor([ent_mask], dtype=torch.long),
        "word_index": torch.tensor([word_index], dtype=torch.long),
        "tokens": tokens
    }

In [70]:
features = build_inference_features(sample_tokens, tokenizer, entity_types)

model.eval()
with torch.no_grad():
    logits_batch, span_indices_batch = model(
        input_ids=features["input_ids"].to(device),
        attention_mask=features["attention_mask"].to(device),
        text_mask=features["text_mask"].to(device),
        ent_mask=features["ent_mask"].to(device),
        word_index=features["word_index"].to(device)
    )

    probs_batch = [torch.sigmoid(logits) for logits in logits_batch]
    pred_batch = model.decoder.decode_batch(probs_batch, span_indices_batch)

print(pred_batch[0])

[{'start': 0, 'end': 2, 'label': 'PS', 'score': 0.9867148995399475}, {'start': 10, 'end': 12, 'label': 'QT', 'score': 0.9557458162307739}, {'start': 5, 'end': 6, 'label': 'LC', 'score': 0.8364378213882446}]


In [71]:
for item in pred_batch[0]:
    s = item["start"]
    e = item["end"]
    label = item["label"]
    score = item["score"]

    span_text = "".join(features["tokens"][s:e+1])
    print(f"{label:>2} | {score:.4f} | ({s}, {e}) | {span_text}")

PS | 0.9867 | (0, 2) | 김도영
QT | 0.9557 | (10, 12) | 4홈런
LC | 0.8364 | (5, 6) | 대구
